In [ ]:
import json
from pathlib import Path
from collections import defaultdict
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm import tqdm
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from pathlib import Path
from collections import defaultdict
from torch.utils.data import DataLoader, random_split
from transformers import get_linear_schedule_with_warmup, AutoTokenizer
import torch.nn.functional as F
import json
from typing import List, Dict, Tuple, Optional
import numpy as np
from collections import defaultdict
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from huggingface_hub import hf_hub_download
# stable_hierarchical_stage2_joint_with_sub_threshold_tuning.py
# ----------------------------

import json, torch, numpy as np
from pathlib import Path
from collections import defaultdict
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from scipy.sparse import csr_matrix
from sklearn.metrics import precision_recall_fscore_support
import torch
from sklearn.metrics import f1_score, precision_score, recall_score

In [ ]:
class AspectAttention(nn.Module):
    def __init__(self, hidden_dim, aspect_emb_dim, d_k=256, d_v=256):
        super().__init__()
        self.Wq = nn.Linear(aspect_emb_dim, d_k, bias=False)
        self.Wk = nn.Linear(hidden_dim, d_k, bias=False)
        self.Wv = nn.Linear(hidden_dim, d_v, bias=False)
        self.Wo = nn.Linear(d_v, hidden_dim)

    def forward(self, enc, aspect_emb, attention_mask):
        """
        enc: (B, L, H)
        aspect_emb: (B, N, E)  <-- Now expects 3D input (Standard)
        """
        # Linear layers handle the (B, N) dimensions automatically
        Q = self.Wq(aspect_emb)          # (B, N, d_k)
        K = self.Wk(enc)                 # (B, L, d_k)
        V = self.Wv(enc)                 # (B, L, d_v)

        # Compute Scores: (B, N, d_k) x (B, d_k, L) -> (B, N, L)
        scores = torch.matmul(Q, K.transpose(1, 2)) / (Q.size(-1) ** 0.5)

        # Masking
        mask = attention_mask.squeeze(-1).unsqueeze(1) # (B, 1, L)
        scores = scores.masked_fill(mask == 0, -1e4) # Safe -1e4
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)  # (B, N, d_v)

        return self.Wo(context)   #(B, N, H)


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def visualize_label_distinctness(model, label_names=None):
    """
    Plots the Cosine Similarity Matrix of the model's learned query vectors.

    Args:
        model: Your PyTorch model (must have model.head_query)
        label_names: List of strings (e.g., ['Price', 'Battery', 'Screen'])
    """
    # 1. Get the query vectors & remove batch dim if present
    # Shape becomes (Num_Labels, Hidden_Dim)
    with torch.no_grad():
        q = model.head_query.detach().cpu()
        if q.dim() == 3:
            q = q.squeeze(0)

    # 2. Normalize (The critical step!)
    q_norm = F.normalize(q, p=2, dim=1)

    # 3. Compute Similarity Matrix (Dot product of normalized vectors)
    # Shape: (Num_Labels, Num_Labels)
    similarity_matrix = torch.mm(q_norm, q_norm.t()).numpy()

    # If no labels provided, use generic indices
    if label_names is None:
        label_names = [f"L{i}" for i in range(len(similarity_matrix))]


    # 5. Print a quick text summary for terminal users
    avg_off_diag = (np.sum(np.abs(similarity_matrix)) - len(similarity_matrix)) / (len(similarity_matrix)**2 - len(similarity_matrix))
    print(f"Average Off-Diagonal Similarity: {avg_off_diag:.4f}")
    if avg_off_diag > 0.5:
        print("WARNING: High overlap detected. Your aspect heads are collapsing!")
    else:
        print("SUCCESS: Vectors are distinct.")


In [ ]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8, disable_torch_grad_focal_loss=True):
        super(AsymmetricLoss, self).__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.disable_torch_grad_focal_loss = disable_torch_grad_focal_loss
        self.eps = eps

    def forward(self, x, y):
        # x: Raw Logits (before Sigmoid)
        # y: Multi-hot targets (0 or 1)

        # Calculate probabilities
        x_sigmoid = torch.sigmoid(x)
        xs_pos = x_sigmoid
        xs_neg = 1 - x_sigmoid

        # Asymmetric Clipping
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)

        # Basic CE calculation
        los_pos = y * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1 - y) * torch.log(xs_neg.clamp(min=self.eps))
        loss = -1 * (self.gamma_pos * los_pos + self.gamma_neg * los_neg)

        # Focal Weighting
        if self.disable_torch_grad_focal_loss:
            torch.set_grad_enabled(False)
        pt0 = xs_pos * y
        pt1 = xs_neg * (1 - y)  # pt = p if t=1 else 1-p
        pt = pt0 + pt1
        one_sided_gamma = self.gamma_pos * y + self.gamma_neg * (1 - y)
        one_sided_w = torch.pow(1 - pt, one_sided_gamma)
        if self.disable_torch_grad_focal_loss:
            torch.set_grad_enabled(True)

        loss *= one_sided_w
        return loss.sum()

In [ ]:
def load_tokenizer_and_model_from_hf(cfg, top_dim, sub_dim, device):
    hf_repo_id = cfg.hf_repo
    hf_subfolder = cfg.hf_encoder_subfolder

    checkpoint_filename = "final_stage1_v6/best_stage1.pt"
    print(f"🔹 Loading Tokenizer & Base Architecture from: {hf_repo_id} (subfolder: {hf_subfolder})")

    try:
        tokenizer = AutoTokenizer.from_pretrained(hf_repo_id, subfolder=hf_subfolder)
        print(f"Tokenizer loaded.")
    except Exception as e:
        print(f"Failed to load tokenizer: {e}")
        tokenizer = AutoTokenizer.from_pretrained(hf_repo_id)

    print(f" Initializing Base Architecture from: {hf_repo_id} (subfolder: {hf_subfolder})")

    # Initialize JointModel with YOUR domain encoder as the base
    model = HierarchicalClassifier(hf_repo_id,
            subfolder=hf_subfolder,
            top_dim=top_dim,
            sub_dim=sub_dim,
            dropout=0.2).to(device)

    print(f" Downloading weights from: {hf_repo_id}/{checkpoint_filename} ...")
    try:
        # Download the file to local cache
        cached_path = hf_hub_download(
            repo_id=hf_repo_id,
            filename=checkpoint_filename)

        # Load into PyTorch
        checkpoint = torch.load(cached_path, map_location=device)

        # Handle the dictionary structure (since you saved {model_state_dict, thresholds, config})
        if "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint # Fallback if it was saved as direct state_dict

        # Load weights into model
        missing, unexpected = model.load_state_dict(state_dict, strict=False)

        print(f"✅ Trained weights loaded successfully!")
        if missing: print(f"   Missing keys: {len(missing)} (Ensure this matches expectation)")
        if unexpected: print(f"   Unexpected keys: {len(unexpected)}", unexpected)

    except Exception as e:
        print(f"Critical Error: Could not load checkpoint from Hugging Face.")

    return tokenizer, model

In [ ]:
def class_wise_eval(y_true, y_pred_probs):
    n_classes = y_true.shape[1]
    print("-" * 85)
    print(f"{'Class ID':<10} | {'Threshold':<10} | {'F1':<10} | {'Precision':<10} | {'Recall':<10} | {'Support':<10}")
    print("-" * 85)
    final_avg_thresholds = np.array([0.538, 0.594, 0.559, 0.568, 0.633, 0.727, 0.492, 0.245, 0.453])
    class_metrics = {}
    macro_f1_scores = []

    for c in range(n_classes):
        # Apply the optimized threshold to the full dataset
        preds_bin = (y_pred_probs[:, c] > final_avg_thresholds[c]).astype(int)
        true_bin = y_true[:, c]

        # Calculate metrics
        precision = precision_score(true_bin, preds_bin, zero_division=0)
        recall = recall_score(true_bin, preds_bin, zero_division=0)
        f1 = f1_score(true_bin, preds_bin, zero_division=0)
        support = true_bin.sum() * 9

        macro_f1_scores.append(f1)

        # Store
        class_metrics[c] = {
            "threshold": final_avg_thresholds[c],
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "support": support
        }

        # Print Row
        print(f"{c} | {final_avg_thresholds[c]:.3f}      | {f1:.4f}     | {precision:.4f}     | {recall:.4f}     | {int(support)}")

    print("-" * 85)
    avg_f1 = np.mean(macro_f1_scores)
    print(f"✅ Final Macro F1 Score: {avg_f1:.4f}")

    return final_avg_thresholds, avg_f1, class_metrics

In [ ]:


# --------------------------
# Config
# --------------------------
class Config:
    output_dir = Path(r"/content/stage2_joint")  # Hugging Face repo or local path
    hf_repo = "Faisal191/aspect-classifier"   # repo with tokenizer/encoder or fallback to yangheng model
    hf_encoder_subfolder = "Domain_trained_encoder" # Specify the subfolder separately
    local_hf_checkpoint = None                # optional path to stage2_compact.pt local copy (if you uploaded)
    hf_compact_path_in_repo= "stage2_full_checkpoint/latest_checkpoint.pt"  # where you uploaded compact ckpt
    data_path = Path(r"/content/final_aspa_data_hierarchical_with_sentiments_temp_v6.json")  # your prepared hierarchical JSON
    epochs = 15
    batch_size = 16
    lr_encoder = 1e-5
    lr_heads = 1e-4
    max_len = 256
    val_split = 0.1
    use_amp = True
    freeze_encoder_epochs = 0
    unfreeze_last_layers_epoch = 0
    unfreeze_full_epoch = 0
    use_sampler = False
    top_k_train = 3
    top_k_eval = 3
    prob_transfer_weight = 0.4
    gating_mode = "mul"
    clip_grad_norm = 1.0
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = 42

    # New configs for threshold tuning & hierarchical masking
    SUB_THRESHOLD_SEARCH = np.linspace(0.05, 0.8, 31)  # candidates to search when fine-tuning sub thresholds
    USE_HIER_MASK_IN_EVAL = True  # apply hierarchical mask to sub predictions at eval/inference
    SAVE_THRESHOLDS_PATH = Path("/content/sub_thresholds.npy")

cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)

# --------------------------
# Repro / Device
# --------------------------
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
torch.backends.cudnn.benchmark = True
DEVICE = cfg.device
print("Device:", DEVICE)

# --------------------------
# Utilities
# --------------------------
def csr_to_torch_sparse(csr):
    coo = csr.tocoo()
    if coo.nnz == 0:
        indices = torch.empty((2,0), dtype=torch.long)
        values = torch.empty((0,), dtype=torch.float32)
    else:
        indices = torch.tensor(np.array([coo.row, coo.col]), dtype=torch.long)
        values = torch.tensor(coo.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(indices, values, torch.Size(coo.shape))

def compute_metrics_numpy(y_true, y_pred, threshold=0.5):
    y_pred_bin = (y_pred > threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_bin, average="macro", zero_division=0
    )
    return precision, recall, f1

def topk_threshold_schedule(epoch:int, total_epochs:int, start=0.35, target=0.56, accel_epoch=4):
    if epoch <= accel_epoch:
        return start + (target - start) * (epoch/accel_epoch)
    else:
        remain = (epoch - accel_epoch) / max(1, total_epochs - accel_epoch)
        return target + 0.02 * remain

def get_alpha_beta(epoch:int, total_epochs:int):
    alpha = 1.0 - 0.1 * (epoch / total_epochs)  # 1.0 → 0.9
    beta = 1.5 + 0.5 * (epoch / total_epochs)  # 1.5 → 2.0
    return round(alpha, 3), round(beta, 3)

def weighted_focal_loss(inputs: torch.Tensor, targets: torch.Tensor, weights: torch.Tensor = None,
                        alpha: float = 0.25, gamma: float = 2.0, eps: float = 1e-8):
    # Convert logits to probabilities safely
    p = torch.sigmoid(inputs)
    p = p.clamp(min=eps, max=1. - eps)

    # Compute the focal modulation for each element
    ce_loss = - (targets * torch.log(p) + (1 - targets) * torch.log(1 - p))
    pt = targets * p + (1 - targets) * (1 - p)  # p_t term
    focal_factor = (1 - pt) ** gamma
    # Apply alpha-balancing
    alpha_factor = targets * alpha + (1 - targets) * (1 - alpha)
    # Combine everything
    loss = alpha_factor * focal_factor * ce_loss
    # Apply class/sample weighting if provided
    if weights is not None:
        loss = loss * weights
    return loss.mean()


def topk_predictions_from_logits(top_logits: np.ndarray, k:int):
    idx = np.argpartition(-top_logits, kth=min(k, top_logits.shape[1]-1), axis=1)[:, :k]
    mask = np.zeros_like(top_logits, dtype=np.int32)
    rows = np.arange(top_logits.shape[0])[:, None]
    mask[rows, idx] = 1
    return mask

# --------------------------
# Dataset
# --------------------------
class HierarchicalDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item.get("text","")
        enc = self.tokenizer(text, truncation=True, padding="max_length",
                             max_length=self.max_len, return_tensors="pt")
        enc = {k:v.squeeze(0) for k,v in enc.items()}
        top_ids = np.array(item["top_cluster_ids"], dtype=np.float32)
        sub_ids = np.array(item["sub_cluster_ids"], dtype=np.float32)
        sub_sparse = csr_to_torch_sparse(csr_matrix(sub_ids.reshape(1,-1)))
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "top_labels": torch.tensor(top_ids, dtype=torch.float32),
            "sub_labels_sparse": sub_sparse
        }

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "top_labels": torch.stack([b["top_labels"] for b in batch]),
        "sub_labels_sparse": [b["sub_labels_sparse"] for b in batch]
    }

# --------------------------
# Model
# --------------------------

class HierarchicalClassifier(nn.Module):
    def __init__(self, encoder_name, top_dim, sub_dim, dropout=0.2, subfolder=None): # Add subfolder
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name, subfolder=subfolder) # Pass subfolder
        self.thresholds = torch.tensor([0.5]*10, dtype=torch.float32)
        h_dim = self.encoder.config.hidden_size
        self.top_dim = top_dim
        self.aspect_emb_dim = h_dim
        self.head_aspect_attention = AspectAttention(
            hidden_dim=h_dim, aspect_emb_dim=self.aspect_emb_dim)
        self.head_query = nn.Parameter(torch.randn(1, top_dim, h_dim))
        self.head_query = nn.Parameter(torch.empty(1, top_dim, h_dim))
        nn.init.normal_(self.head_query, mean=0, std=0.02)
        self.head_gate_proj = nn.Linear(h_dim * 2, h_dim * 2)
        # Stage 1 (top-level)
        self.head_norm = nn.LayerNorm(h_dim)
        self.head_weight = nn.Parameter(torch.randn(top_dim, h_dim))
        self.head_bias = nn.Parameter(torch.zeros(top_dim))

        self.top_loss_fn = AsymmetricLoss(
                              gamma_neg=2,    # Standard penalty
                              gamma_pos=1,
                              clip=0.0)        # No clipping (We want to learn from negatives here)"""

        # Initialization
        torch.nn.init.normal_(self.head_weight, std=0.02)
        self.head_dropout = nn.Dropout(dropout)

        # Stage 2 (sub-level)
        self.sub_head = nn.Linear(h_dim + top_dim, sub_dim)
        #self.dropout = nn.Dropout(dropout)

        # Mapping-related attributes
        self.top_to_sub_map = None          # sparse tensor (top_dim × sub_dim)
        self.top_to_sub_dense = None        # dense cache for matmul
        self.top_dim = top_dim
        self.sub_dim = sub_dim

    # ----------------------------------------------------------
    def set_top_to_sub_map(self, mapping: dict):
        self.top_to_sub_dict = mapping  # keep for loss computation

        rows, cols = [], []
        for t, subs in mapping.items():
            rows.extend([t] * len(subs))
            cols.extend(subs)

        indices = torch.tensor([rows, cols], dtype=torch.long)
        values = torch.ones(len(rows), dtype=torch.float32)
        sparse_map = torch.sparse_coo_tensor(indices, values, (self.top_dim, self.sub_dim))
        self.top_to_sub_map = sparse_map.coalesce()
        self.top_to_sub_dense = None


    # ----------------------------------------------------------
    def _get_dense_map(self, device, dtype):
        """
        Safely gets (or builds) dense mapping tensor.
        - Auto moves to correct device/dtype (handles AMP).
        - Caches result for reuse.
        """
        if self.top_to_sub_dense is None or self.top_to_sub_dense.device != device:
            dense_map = self.top_to_sub_map.to_dense().to(device)
            self.top_to_sub_dense = dense_map
        # ensure dtype matches current autocast precision
        if self.top_to_sub_dense.dtype != dtype:
            self.top_to_sub_dense = self.top_to_sub_dense.to(dtype)
        return self.top_to_sub_dense

    # ----------------------------------------------------------
    def forward(self, input_ids, attention_mask, gating_mode="add", prob_weight=0.5):
        # Encoder
        enc = self.encoder(input_ids, attention_mask=attention_mask).last_hidden_state
        batch_size = enc.size(0)
        mask = attention_mask.unsqueeze(-1).float() #(B, L, 1)
        pooled = (enc * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        top_query_expanded = self.head_query.expand(batch_size, -1, -1) # (B, N, H)
        aspect_context = self.head_aspect_attention(enc, top_query_expanded, mask) # (B, N, H)
        pooled_expanded = pooled.unsqueeze(1).expand(-1, self.top_dim, -1) # (B, N, H)
        # 3. Gating Logic
        # Concatenate: (B, N, 2*H)
        combined = torch.cat([pooled_expanded, aspect_context], dim=2)
        # Compute Gate Alpha: (B, N, H)
        alpha = torch.sigmoid(self.head_gate_proj(combined))
        alpha_pooled, alpha_context = alpha.chunk(2, dim=-1)

        # Fuse: Alpha controls how much we use Pooled vs Attention
        fused_embedding = alpha_pooled * pooled_expanded + alpha_context * aspect_context
        fused_norm = self.head_norm(self.head_dropout(fused_embedding))

        # 4. Top-Level Prediction
        top_logits = (fused_norm * self.head_weight).sum(dim=-1) + self.head_bias

        """p_top = torch.sigmoid(top_logits)

        # Compute sub-cluster prior
        batch_size, device, dtype = p_top.size(0), p_top.device, p_top.dtype
        if self.top_to_sub_map is not None:
            map_dense = self._get_dense_map(device, dtype)
            sub_prior = torch.matmul(p_top, map_dense)
            sub_prior.clamp_(0.0, 1.0)
        else:
            sub_prior = torch.zeros(batch_size, self.sub_dim, device=device, dtype=dtype)

        # Move thresholds to the same device as p_top
        hard_mask = (p_top >= self.thresholds.to(p_top.device)).float()
        concat = torch.cat([pooled, hard_mask], dim=1)
        sub_logits = self.sub_head(concat)


        # Hierarchical gating
        if gating_mode == "add":
            sub_logits = sub_logits + prob_weight * sub_prior
        elif gating_mode == "mul":
            sub_logits = sub_logits * (1.0 + prob_weight * sub_prior)"""

        return top_logits#, sub_logits

    def compute_loss(self, top_logits, y_top, y_sub_sparse_list,
                     alpha=0.9, beta=1.1, top_k=3, hierarchical_reg_weight=0.1):
        device = top_logits.device
        y_top = y_top.to(dtype=torch.float32, device=device)
        top_loss = F.binary_cross_entropy_with_logits(top_logits, y_top, pos_weight=self.top_pos_weight.to(device) if self.top_pos_weight is not None else None)
        #top_loss = self.top_loss_fn(top_logits, y_top)
        total_loss = top_loss

        """y_sub_dense = torch.stack([s.to_dense().squeeze(0).to(device) for s in y_sub_sparse_list])
        with torch.no_grad():
            _, topk_idx = torch.topk(top_logits, k=min(top_logits.size(1), top_k), dim=1)
            mask = torch.zeros_like(y_sub_dense)
            for i in range(y_top.size(0)):
                chosen_tops = topk_idx[i].tolist()
                relevant_subs = set()
                for t in chosen_tops:
                    if hasattr(self, "top_to_sub_dict") and t in self.top_to_sub_dict:
                        relevant_subs.update(self.top_to_sub_dict[t])
                if len(relevant_subs) > 0:
                    mask[i, list(relevant_subs)] = 1.0
            if mask.sum() == 0: mask = torch.ones_like(mask)

        if self.sub_pos_weight is not None:
            pos_w = self.sub_pos_weight.to(device).unsqueeze(0).repeat(y_sub_dense.size(0),1)
            sub_weights = pos_w * mask
        else:
            sub_weights = mask

        sub_loss = weighted_focal_loss(sub_logits, y_sub_dense, weights=sub_weights)

        # hierarchical reg
        p_top = torch.sigmoid(top_logits).detach()
        p_sub = torch.sigmoid(sub_logits)"""

        #total_loss = top_loss alpha* + beta*sub_loss
        top_loss = top_loss.detach()
        #sub_loss = sub_loss.detach()
        return total_loss, float(top_loss)#, float(sub_loss)

# --------------------------
# Evaluate
# --------------------------
def apply_hierarchical_mask_to_sub_probs(p_top, p_sub, top_thresholds, top_to_sub_map):
    """
    p_top: (N, T) numpy array of top probabilities
    p_sub: (N, S) numpy array of sub probabilities
    top_thresholds: array-like length T
    top_to_sub_map: one of:
        - dict: {top_idx: [sub_idx, ...]}
        - torch.sparse_coo_tensor or torch.Tensor (dense)
        - scipy.sparse matrix
        - numpy.ndarray (dense)
    Returns: p_sub masked (numpy array)
    """
    import numpy as np
    import torch
    from scipy.sparse import issparse as is_scipy_sparse

    if top_to_sub_map is None or top_thresholds is None:
        return p_sub

    # binarize top predictions according to per-class thresholds
    top_thresholds = np.array(top_thresholds)
    top_pred_bin = (p_top > top_thresholds[None, :]).astype(np.int32)  # (N, T)

    # Build a dense top->sub adjacency matrix (T x S) in numpy
    dense_map = None
    if isinstance(top_to_sub_map, dict):
        # infer S from p_sub shape
        T = top_pred_bin.shape[1]
        S = p_sub.shape[1]
        dense_map = np.zeros((T, S), dtype=np.int8)
        for t, subs in top_to_sub_map.items():
            # ensure ints and valid indices
            if len(subs) == 0:
                continue
            dense_map[int(t), np.array(subs, dtype=np.int64)] = 1
    else:
        # assume tensor / sparse / numpy
        # Torch sparse: convert to dense then numpy
        if isinstance(top_to_sub_map, torch.Tensor):
            # handle sparse or dense torch tensor
            try:
                if top_to_sub_map.is_sparse:
                    dense_map = top_to_sub_map.to_dense().cpu().numpy()
                else:
                    dense_map = top_to_sub_map.cpu().numpy()
            except Exception:
                # fallback: coalesce then to_dense
                dense_map = top_to_sub_map.coalesce().to_dense().cpu().numpy()
        elif is_scipy_sparse(top_to_sub_map):
            dense_map = top_to_sub_map.toarray()
        else:
            # hope it's convertible (e.g., numpy array)
            dense_map = np.asarray(top_to_sub_map)

    # Sanity: ensure shapes align
    T = top_pred_bin.shape[1]
    S = p_sub.shape[1]
    if dense_map.shape != (T, S):
        # try to transpose if user stored (S, T)
        if dense_map.shape == (S, T):
            dense_map = dense_map.T
        else:
            raise ValueError(f"top_to_sub_map shape mismatch: expected ({T},{S}), got {dense_map.shape}")

    # compute mask: for each sample i, mark subs that have any active parent
    # top_pred_bin (N x T) dot dense_map (T x S) -> counts (N x S)
    mask_counts = top_pred_bin.dot(dense_map)  # int counts
    mask = (mask_counts > 0).astype(float)     # 1.0 for allowed subs, 0.0 otherwise

    # If no top predicted for an example, choose fallback: allow all subs (original behavior)
    # (original code set mask[i,:]=1.0 if relevant_subs empty)
    no_top = (top_pred_bin.sum(axis=1) == 0)
    if no_top.any():
        mask[no_top, :] = 1.0

    return p_sub * mask

def evaluate(model, dataloader, device, epoch, top_thresholds=None, sub_thresholds=None, sub_threshold=0.5, use_hier_mask=True):
    model.eval()
    all_y_top, all_p_top, all_y_sub, all_p_sub = [], [], [], []
    all_p_top_cont, all_p_sub_cont = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Eval E{epoch}"):
            ids, mask = batch["input_ids"].to(device), batch["attention_mask"].to(device)
            y_top = batch["top_labels"].cpu().numpy()
            y_sub = np.stack([s.to_dense().squeeze(0).cpu().numpy() for s in batch["sub_labels_sparse"]])
            top_logits = model(ids, mask)
            p_top = torch.sigmoid(top_logits).cpu().numpy()
            """p_sub = torch.sigmoid(sub_logits).cpu().numpy()

            # apply hierarchical mask if requested (use continuous probs)
            if use_hier_mask and top_thresholds is not None:
                p_sub_masked = apply_hierarchical_mask_to_sub_probs(p_top, p_sub, top_thresholds, model.top_to_sub_map)
            else:
                p_sub_masked = p_sub"""

            # --- convert top logits using per-class thresholds ---
            if top_thresholds is not None:
                # Ensure top_thresholds is a numpy array before comparison
                p_top_bin = (p_top > np.array(top_thresholds)).astype(int)
            else:
                p_top_bin = (p_top > 0.5).astype(int)

            # For sub predictions, we allow per-class thresholds (sub_thresholds) if provided
            """if sub_thresholds is not None:
                # convert per-class
                p_sub_bin = (p_sub_masked > np.array(sub_thresholds)[None,:]).astype(int)
            else:
                p_sub_bin = (p_sub_masked > sub_threshold).astype(int)"""

            all_y_top.append(y_top); all_p_top.append(p_top_bin)
            #all_y_sub.append(y_sub); all_p_sub.append(p_sub_bin)
            all_p_top_cont.append(p_top); #all_p_sub_cont.append(p_sub_masked)

    y_top = np.vstack(all_y_top); p_top = np.vstack(all_p_top)
    p_top_cont = np.vstack(all_p_top_cont)
    class_wise_eval(y_top, p_top_cont)
    #y_sub = np.vstack(all_y_sub); p_sub = np.vstack(all_p_sub)
    p_top_th, r_top_th, f1_top = compute_metrics_numpy(y_top, p_top)
    #p_sub_th, r_sub_th, f1_sub = compute_metrics_numpy(y_sub, p_sub)
    # hier_acc definition: both top & sub have at least one correct label in example
    # using continuous predictions for top/sub to avoid binarization artifacts
    # p_sub_cont = np.vstack(all_p_sub_cont)
    #hier_acc = np.mean((((y_top * (p_top_cont > 0.5)).sum(1) > 0) & ((y_sub * (p_sub_cont > 0.5)).sum(1) > 0)))
    return {"top_f1":f1_top,
            "p_top":p_top_th,"r_top":r_top_th,
             "y_top": y_top, "p_top_cont": p_top_cont}

# --------------------------
# Threshold fine-tuning helpers
# --------------------------
# --------------------------
# Train loop (improved)
# --------------------------
def train(cfg):
    data = json.load(open(cfg.data_path))
    top_to_sub_map = defaultdict(set)
    for it in data:
        for k,v in it.get("top_to_sub_ids", {}).items():
            top_to_sub_map[int(k)].update(v)
    top_to_sub_map = {k:list(v) for k,v in top_to_sub_map.items()}
    print(top_to_sub_map)

    TOP, SUB = len(data[0]["top_cluster_ids"]), len(data[0]["sub_cluster_ids"])
    tokenizer, model = load_tokenizer_and_model_from_hf(cfg, TOP, SUB, DEVICE)
    dataset = HierarchicalDataset(data, tokenizer, max_len=cfg.max_len)
    val_size = int(cfg.val_split*len(dataset))
    train_data, val_data = random_split(dataset, [len(dataset)-val_size, val_size])
    np.save(cfg.output_dir/"valid_indices.npy", np.array(val_data.indices))

    train_loader = DataLoader(train_data, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_data, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

    # Model init
    model.set_top_to_sub_map(top_to_sub_map)

    # Load thresholds for top and sub clusters
    top_thresholds = np.array([0.538, 0.594, 0.559, 0.568, 0.633, 0.727, 0.492, 0.245, 0.453])
    sub_thresholds = np.random.uniform(0.35, 0.55, 29)  # assuming 29 sub-clusters
    print("Initial thresholds loaded.")

    metrics = evaluate(model, val_loader, DEVICE, epoch=8,
            top_thresholds=top_thresholds,
            sub_thresholds=sub_thresholds,
            use_hier_mask=cfg.USE_HIER_MASK_IN_EVAL)

    # Compute pos weights
    top_counts = np.sum([it["top_cluster_ids"] for it in data], axis=0)
    sub_counts = np.sum([it["sub_cluster_ids"] for it in data], axis=0)
    model.top_pos_weight = torch.tensor((len(data)-top_counts)/(top_counts+1e-6),dtype=torch.float32,device=DEVICE)
    model.sub_pos_weight = torch.tensor((len(data)-sub_counts)/(sub_counts+1e-6),dtype=torch.float32,device=DEVICE)

    encoder_params = []
    new_head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "encoder" in name:
            encoder_params.append(param)
        else:
            new_head_params.append(param)

    optimizer = torch.optim.AdamW([
        {"params": encoder_params,   "lr": cfg.lr_encoder}, # Slower learning for base
        {"params": new_head_params,  "lr": cfg.lr_heads}    # Faster learning for new layers
    ], weight_decay=0.01)

    total_steps = len(train_loader)*cfg.epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)
    scaler = torch.amp.GradScaler(enabled=cfg.use_amp)

    # ------------- Resume if checkpoint exists -------------
    start_epoch, best_sum = 1, 0.0
    ckpt_path = cfg.output_dir / "latest_checkpoint.pt"
    if ckpt_path.exists():
        print("🔄 Resuming from last checkpoint...")
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        sub_thresholds = ckpt.get("sub_thresholds", sub_thresholds)
        start_epoch = ckpt["epoch"] + 1
        best_sum = ckpt.get("best_sum", 0.0)
        print(f"Resumed from epoch {start_epoch-1}. Best combined F1: {best_sum:.4f}")
    # --------------------------------------------------------

    hier_smooth = []

    for epoch in range(start_epoch, cfg.epochs+1):
        if epoch == cfg.unfreeze_last_layers_epoch:
            for name,p in model.encoder.named_parameters():
                if "block.6" in name or "block.7" in name: p.requires_grad=True
            print(f"Unfroze last layers at epoch {epoch}.")
        if epoch == cfg.unfreeze_full_epoch:
            for p in model.encoder.parameters(): p.requires_grad=True
            print(f"Unfroze full encoder at epoch {epoch}.")

        top_thresh = topk_threshold_schedule(epoch, cfg.epochs, start=0.32, target=0.56, accel_epoch=4)
        alpha, beta = get_alpha_beta(epoch, cfg.epochs)
        model.train()
        for batch in tqdm(train_loader, desc=f"Train E{epoch}"):
            ids, mask = batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE)
            y_top, y_sub_sparse = batch["top_labels"].to(DEVICE), batch["sub_labels_sparse"]

            with torch.amp.autocast(device_type='cuda', enabled=cfg.use_amp):
                top_logits = model(ids, mask, cfg.gating_mode, cfg.prob_transfer_weight)
                loss,_ = model.compute_loss(top_logits, y_top, y_sub_sparse,
                               alpha=alpha, beta=beta, top_k=cfg.top_k_train, hierarchical_reg_weight=0.08)

            if not torch.isfinite(loss):
                print(f"⚠️ Skipping batch due to non-finite loss at loss {loss}")
                optimizer.zero_grad()
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        # NaN-safe threshold clipping
        sub_thresholds = np.nan_to_num(sub_thresholds, nan=0.45, posinf=0.5, neginf=0.4)
        sub_thresholds = np.clip(sub_thresholds, 0.35, 0.6)

        # Eval
        print(f"Epoch {epoch}")
        metrics = evaluate(model, val_loader, DEVICE, epoch,
                   top_thresholds=top_thresholds,
                   sub_thresholds=sub_thresholds,
                   use_hier_mask=cfg.USE_HIER_MASK_IN_EVAL)


        # Save best by F1 sum
        f1_sum = metrics["top_f1"] #+ metrics["sub_f1"]
        if f1_sum > best_sum:
            best_sum = f1_sum
            torch.save(model.state_dict(), cfg.output_dir/"best_stage1_v6.pt")
            print("💾 Saved new best model (F1 sum improved).")
        visualize_label_distinctness(model)

        # Periodic checkpoint
        if epoch % 3 == 0:
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "sub_thresholds": sub_thresholds,
                "best_sum": best_sum,
            }, ckpt_path)
            print(f"📦 Checkpoint saved at epoch {epoch}.")

    print(f"✅ Training finished. Best combined F1: {best_sum:.4f}")

if __name__=="__main__":
    train(cfg)

In [ ]:
"""# Used to securely store your API key
from google.colab import userdata
from huggingface_hub import HfApi
from pathlib import Path

# Retrieve the token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Initialize HfApi with the token
api = HfApi(token=hf_token)

username = "Faisal191"
repo_name = "aspect-classifier"
repo_id = f"{username}/{repo_name}"

checkpoint_path = Path(r"/content/stage2_joint/best_stage1_v6.pt")

# Attempt the upload again with proper authentication
api.upload_file(
    path_or_fileobj=checkpoint_path,
    path_in_repo="final_stage1_v6/best_stage1.pt",
    repo_id=repo_id,
    commit_message="Upload final checkpoint",
)

print(f"✅ Uploaded checkpoint to https://huggingface.co/{repo_id}/blob/main/final_absa_checkpoint/epoch5.pt")"""
